# 🦞 OpenClaw + Ollama (ローカルLLM) on Google Colab

このノートブックでは **ngrokなし・APIキーなし** でOpenClawをローカルLLM（Ollama）で動かします。

| 項目 | 内容 |
|------|------|
| LLM | qwen2.5:7b-instruct-q4_K_M（Ollama経由） |
| GPU | T4（Colab無料枠） |
| 外部サービス | 不要 |

---
> ⚠️ **事前確認**：「ランタイム → ランタイムのタイプを変更 → **T4 GPU**」を選択してから実行してください

## ✅ Step 1：GPU確認

In [ ]:
!nvidia-smi
# T4 GPU が表示されればOK
# 表示されない場合は「ランタイム → ランタイムのタイプを変更 → T4 GPU」に変更してください

## 🟢 Step 2：Node.js 24 インストール

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_24.x | bash -
!apt-get install -y nodejs -qq
!echo "Node.js version: $(node --version)"
!echo "npm version: $(npm --version)" 

## 🦙 Step 3：Ollama インストール・起動・モデルDL

Ollamaはローカル LLM を動かすためのランタイムです。インストール・サーバー起動・モデルDLを一括で行います。

In [ ]:
# zstd と pciutils をインストール（Ollamaに必要）
!apt-get install -y zstd pciutils -qq

# Ollama インストール
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

# インストールスクリプトが起動したプロセスをいったん終了
subprocess.run("pkill -f 'ollama serve'", shell=True)
time.sleep(2)

# バックグラウンドで起動
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

# 起動を最大30秒待つ
for i in range(30):
    try:
        requests.get("http://localhost:11434", timeout=1)
        print(f"\n✅ Ollama サーバー起動完了 (pid={proc.pid})")
        break
    except Exception:
        print(f"  待機中... {i+1}秒", end="\r")
        time.sleep(1)
else:
    print("❌ 起動失敗。ログ確認:")
    !cat /tmp/ollama.log

In [ ]:
# モデルのダウンロード（約3〜5分）
!ollama pull qwen2.5:7b-instruct-q4_K_M

In [ ]:
# 動作確認
!ollama run qwen2.5:7b-instruct-q4_K_M "Hello! Reply in one sentence." --nowordwrap

## 🦞 Step 4：OpenClaw インストール＆設定

In [ ]:
!npm install -g openclaw@latest
!openclaw --version

In [ ]:
import os, json

os.makedirs("/root/.openclaw", exist_ok=True)
os.makedirs("/root/.openclaw/agents/main/agent", exist_ok=True)

# openclaw.json5（グローバル設定）
config = {
    "models": {
        "providers": {
            "ollama": {
                "baseUrl": "http://localhost:11434",
                "defaultModel": "qwen2.5:7b-instruct-q4_K_M",
                "parameters": {
                    "numCtx": 32768
                }
            }
        },
        "default": "ollama"
    }
}
with open("/root/.openclaw/openclaw.json5", "w") as f:
    json.dump(config, f, indent=2)

# models.json（エージェント別モデル設定）
models = {
    "default": "ollama/qwen2.5:7b-instruct-q4_K_M",
    "fallbacks": [],
    "providers": {
        "ollama": {
            "baseUrl": "http://127.0.0.1:11434",
            "api": "ollama",
            "models": [
                {
                    "id": "qwen2.5:7b-instruct-q4_K_M",
                    "name": "qwen2.5:7b-instruct-q4_K_M",
                    "reasoning": False,
                    "input": ["text"],
                    "cost": {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
                    "contextWindow": 32768,
                    "maxTokens": 8192
                }
            ],
            "apiKey": "OLLAMA_API_KEY"
        }
    }
}
with open("/root/.openclaw/agents/main/agent/models.json", "w") as f:
    json.dump(models, f, indent=2)

os.environ["OLLAMA_API_KEY"] = "ollama-local"
print("✅ OpenClaw 設定完了")

## 🚀 Step 5：OpenClaw ゲートウェイ起動

In [ ]:
import subprocess, time, os

env = os.environ.copy()
env["OLLAMA_API_KEY"] = "ollama-local"

proc = subprocess.Popen(
    ["openclaw", "gateway", "--port", "18789"],
    stdout=open("/tmp/openclaw.log", "w"),
    stderr=subprocess.STDOUT,
    env=env
)
time.sleep(5)
print("✅ OpenClaw ゲートウェイ起動完了 (port: 18789)")
print("ログ確認: !cat /tmp/openclaw.log")

## 🖥️ Step 6：Colab内でダッシュボードを表示

ngrokは不要です。Colab組み込みの機能でブラウザ内に表示します。

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(18789, height=700)
# ↑ このセルの出力にダッシュボードが表示されます

In [ ]:
# iframeで表示されない場合はこちら（新しいタブで開く）
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(18789)

## 🎮 Step 7：CLIデモ — エージェントにメッセージを送る

OpenClaw CLIから直接エージェントに指示を出せます。

In [ ]:
# デモ①：エージェント一覧を確認
!OLLAMA_API_KEY=ollama-local openclaw agents list

In [ ]:
# デモ②：エージェント(main)の詳細確認
!OLLAMA_API_KEY=ollama-local openclaw agents show main

In [ ]:
# デモ③：エージェントにメッセージを送信
!OLLAMA_API_KEY=ollama-local openclaw agent --agent main --message "こんにちは！自己紹介してください。" 

In [ ]:
# デモ④：自由にメッセージを送る（messageを書き換えてください）
import subprocess, os

message = "Pythonでフィボナッチ数列を10個出力するコードを教えてください。"  # ← ここを自由に変えてください

result = subprocess.run(
    ["openclaw", "agent", "--agent", "main", "--message", message],
    capture_output=True, text=True,
    env={**os.environ, "OLLAMA_API_KEY": "ollama-local"}
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

## 💡 Tips & トラブルシューティング

| 症状 | 対処法 |
|------|--------|
| GPUが割り当てられない | ランタイム → ランタイムのタイプを変更 → T4 GPU |
| Ollamaが応答しない | Step 3を再実行 |
| iframeが空白 | Step 5を再実行し10秒待つ |
| セッション切れ | Step 3〜5を再実行（Step 1・2は不要）|
| `No API key for anthropic` | Step 4の設定セルを再実行 |
| `unknown option` エラー | `--agent` と `main` を別引数にする（文字列結合しない） |

In [ ]:
# 診断：インストール済みモデルとOpenClaw状態確認
print("=== インストール済みモデル ===")
!ollama list
print("\n=== OpenClaw 診断 ===")
!OLLAMA_API_KEY=ollama-local openclaw doctor